## Summary

Create numpy files for velocities, acceleration and particle relationships.

## Imports and set-up

In [1]:
import sys
from pathlib import Path
import os

In [2]:
import numpy as np

In [3]:
project_working_dir = str(Path(sys.path[0]).parent)
sys.path += [project_working_dir]
os.chdir(project_working_dir)

In [4]:
from src.utils.file_io import read_json
from src.utils.read_obj import parse_obj

from src.simulation.setup.extract_clothing_vertex_data import extract_all_piece_vertices

In [5]:
from src.parameters import AVATAR_SCALING

## Read clothing data and avatar mesh

This is a very simple pattern. Probably too simple for a shirt that you can buy.

In [6]:
clothing_data = read_json('./assets/sewing_shirt.json')

In [7]:
avatar_mesh = parse_obj('./assets/BodyMesh.obj', './assets/BodyAnnotations.json')
avatar_mesh.scale_vertices(AVATAR_SCALING)

In [8]:
dynamic_pieces, sewing_constraints = extract_all_piece_vertices(clothing_data, avatar_mesh)

## Create Regression Inputs

Create inputs to simulate the physics of different operations after one frame

In [9]:
piece = dynamic_pieces["L-1"]

In [10]:
vertices = np.copy(piece.mesh.vertices_3d)

In [11]:
velocity = np.copy(piece.velocity)

In [12]:
np.save("./profiling/numpy/piece_velocity.npy", velocity)

In [13]:
acceleration = np.copy(piece.acceleration)

In [14]:
np.save("./profiling/numpy/piece_acceleration.npy", acceleration)

In [15]:
stress_relations = np.copy(piece.vertex_relations.stress_relations)

In [16]:
np.save("./profiling/numpy/piece_stress_relations.npy", stress_relations)

In [17]:
shear_relations = np.copy(piece.vertex_relations.shear_relations)

In [18]:
np.save("./profiling/numpy/piece_shear_relations.npy", shear_relations)

In [19]:
bend_relations = np.copy(piece.vertex_relations.bend_relations)

In [20]:
np.save("./profiling/numpy/piece_bend_relations.npy", bend_relations)

In [21]:
sewing_constraints.relations[0].from_piece

'L-1'

In [22]:
sewing_constraints.relations[0].to_piece

'L-2'

In [23]:
other_piece = dynamic_pieces["L-2"]
other_vertices = np.copy(other_piece.mesh.vertices_3d)

In [24]:
np.save("./profiling/numpy/piece_other_vertices.npy", other_vertices)

In [25]:
sewing_relation = np.copy(sewing_constraints.relations[0].indices)

In [26]:
np.save("./profiling/numpy/sewing_relation.npy", sewing_relation)

## Create Regression Data

Create expected results after different operations

In [27]:
piece.acceleration *= 0

In [28]:
piece.apply_stress_force()

In [29]:
np.save('./profiling/numpy/expected_piece_acceleration_after_stress.npy', piece.acceleration)

In [30]:
piece.acceleration *= 0

In [31]:
piece.apply_shear_force()

In [32]:
np.save('./profiling/numpy/expected_piece_acceleration_after_shear.npy', piece.acceleration)

In [33]:
piece.acceleration *= 0

In [34]:
piece.apply_bend_force()

In [35]:
np.save('./profiling/numpy/expected_piece_acceleration_after_bend.npy', piece.acceleration)

## Profile each numpy operation

For reference, profile the execution time of each type of operation

Reset the acceleration to gravity

In [16]:
%timeit piece.acceleration *= 0; piece.apply_gravity()

3.28 µs ± 371 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


Change accerlation from stress, add in some offset so stress applies

In [17]:
piece.mesh.scale_vertices(1.05); 

In [18]:
%timeit piece.apply_stress_force()

863 µs ± 52.8 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [21]:
%timeit piece.apply_shear_force()

788 µs ± 17.4 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [20]:
%timeit piece.apply_bend_force()

256 µs ± 16.2 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
